# Assignment 2 - Constrained SQL Generation

Before running: set the runtime to a GPU (Runtime > Change runtime type > T4 GPU) and upload `a2.db` into the
Files panel on the left.

## Setup

In [ ]:
!pip install -q outlines==1.3.3 llguidance==1.9.1 lark==1.3.1 transformers==5.18.0

In [ ]:
from importlib.metadata import version

for lib in ["outlines", "llguidance", "lark", "transformers", "torch"]:
    print(lib, version(lib))

In [ ]:
import os
import sqlite3

assert os.path.exists("a2.db"), "upload a2.db first!"
conn = sqlite3.connect("a2.db")

# print the schema straight from the database
for row in conn.execute("SELECT sql FROM sqlite_master"):
    print(row[0])

## Part A - The grammar

In [ ]:
grammar = r"""
start : query

query : "SELECT" columns "FROM" table where_clause?
columns : "*" | NAME ("," NAME)*
table : "users" | "orders"
NAME : "id" | "name" | "city" | "age" | "user_id" | "total" | "status"

where_clause : "WHERE" condition
condition : and_condition ("OR" and_condition)*
and_condition : comparison ("AND" comparison)*
comparison : NAME OP value

OP : "=" | "!=" | "<=" | ">=" | "<" | ">"

value : FLOAT | INT | STRING
FLOAT : /[0-9]+\.[0-9]+/
INT : /[0-9]+/
STRING : /'[^']*'/

%import common.WS
%ignore WS
"""

How I filled in the TODOs:

- `start : query` - Outlines (llguidance) and Lark both look for a rule called `start`, so I just point it at `query`.
- `condition` is a bunch of `and_condition`s joined by OR, and each `and_condition` is comparisons joined by AND.
  Doing it in two levels makes AND bind tighter than OR, which is how SQL does it.
- `comparison` is always a column name, an operator, and a value, like `age < 30`.
- `OP` has the 6 operators. The lexer takes the longest match, so `<=` is one operator and not `<` then `=`.
- `FLOAT` needs digits on both sides of the dot, `INT` is just digits, and `STRING` is anything inside single quotes.
- `%ignore WS` lets spaces go anywhere, which matters because the model makes tokens like `" FROM"` with the space attached.

### Parse tests

In [ ]:
from lark import Lark
from lark.exceptions import LarkError

parser = Lark(grammar)

tests = [
    ("SELECT name, age FROM users WHERE city = 'Raleigh' AND age < 30",
     "should pass: column list, a string and an int, joined by AND"),
    ("SELECT * FROM orders WHERE total >= 100.50 OR status != 'cancelled'",
     "should pass: star, a float, two-character operators, and OR"),
    ("SELECT COUNT(*) FROM orders",
     "should fail: COUNT isn't one of the allowed column names (no aggregates)"),
    ("SELECT name FROM users WHERE",
     "should fail: WHERE with nothing after it"),
]

for query, why in tests:
    try:
        parser.parse(query)
        print("ACCEPTED:", query)
    except LarkError:
        print("REJECTED:", query)
    print("   ", why)

## Part B - Generate both ways

Both use the same model and the same function call. The only difference is the constrained one gets `CFG(grammar)`,
which makes Outlines mask out every token that would break the grammar before each token is sampled.

In [ ]:
import torch
import outlines
from outlines.types import CFG
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda")
model = outlines.from_transformers(hf_model, tokenizer)

unconstrained = outlines.Generator(model)
constrained = outlines.Generator(model, CFG(grammar))

# the unconstrained one has no logits processor, the constrained one does
print("unconstrained:", unconstrained.logits_processor)
print("constrained:", type(constrained.logits_processor).__name__)

Settings: temperature 0.8, 2 samples per prompt using seeds 1234 and 5678. Qwen's config has its own top_p, top_k and
repetition penalty, so I turned those off so the only setting is temperature.

In [ ]:
SEEDS = [1234, 5678]
settings = dict(do_sample=True, temperature=0.8, top_p=1.0, top_k=0,
                repetition_penalty=1.0, max_new_tokens=128)

schema = """CREATE TABLE users  (id INTEGER, name TEXT, city TEXT, age INTEGER);
CREATE TABLE orders (id INTEGER, user_id INTEGER, total REAL, status TEXT);"""

def make_prompt(question):
    messages = [
        {"role": "system", "content": "Write a SQLite query for this database:\n\n" + schema
                                      + "\n\nReply with a single SQL query and nothing else."},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

### Prompts

In [ ]:
# Group 1: the grammar can answer these
group1 = [
    "Who are all the users that live in Raleigh?",
    "Which orders have a total over 100 dollars?",
    "What are the names of users under 30?",
    "List the id and total of every order that is still pending.",
    "Show the names and ages of users who live in Durham and are older than 40.",
    "Find all orders that were cancelled or cost less than 20 dollars.",
    "Which city does Priya live in?",
    "Show every order placed by the user with id 7.",
]

# Group 2: these need a join, count, average, or sort, so the grammar can't do them
group2 = [
    "How many orders did each user place?",
    "What is the average order total?",
    "What are the names of users who have placed an order over 300 dollars?",
    "List the five most expensive orders, from highest total to lowest.",
]

In [ ]:
results = []

for group, questions in [(1, group1), (2, group2)]:
    for question in questions:
        prompt = make_prompt(question)
        for seed in SEEDS:
            for condition, gen in [("Unconstrained", unconstrained), ("Constrained", constrained)]:
                set_seed(seed)
                sql = gen(prompt, **settings)
                results.append({"condition": condition, "group": group, "seed": seed,
                                "question": question, "sql": sql})
                print(condition, "| group", group, "| seed", seed, "|", repr(sql))

print(len(results), "generations")

## Part C - Measure and look

A query is valid if SQLite accepts `EXPLAIN <query>` on a2.db. This checks the syntax and also that the tables and
columns exist, without actually running the query.

In [ ]:
import pandas as pd

def sqlite_error(sql):
    try:
        conn.execute("EXPLAIN " + sql)
        return None
    except sqlite3.Error as e:
        return str(e)

df = pd.DataFrame(results)
df["error"] = df["sql"].apply(sqlite_error)
df["valid"] = df["error"].isna()

table = df.groupby(["condition", "group"])["valid"].agg(["sum", "count"]).reset_index()
table.columns = ["Condition", "Group", "SQLite accepts", "n"]
table

In [ ]:
# the ones SQLite rejected and why
for _, row in df[~df["valid"]].iterrows():
    print(row["condition"], "| group", row["group"], "|", repr(row["sql"]))
    print("   ->", row["error"])

### Constrained Group 2 queries and what they returned

In [ ]:
for _, row in df[(df["condition"] == "Constrained") & (df["group"] == 2)].iterrows():
    print("Q:", row["question"])
    print("SQL:", row["sql"].strip())
    if row["valid"]:
        rows = conn.execute(row["sql"]).fetchall()
        print("Returned", len(rows), "rows:", rows[:5], "..." if len(rows) > 5 else "")
    else:
        print("SQLite rejected it:", row["error"])
    print()

## Discussion

**What did the constrained model produce for Group 2, and would a user notice it's wrong?**

It never says "I can't do that" because the grammar doesn't let it. It just writes some query that fits:

- "How many orders did each user place?" - it mixed columns from both tables (like `name` from orders), probably
  because it wanted to do a join. SQLite rejected both of these, so the user would at least get an error. These
  were the only constrained queries that failed.
- "What is the average order total?" - it gave `SELECT total FROM orders`, which is just all 50 totals and not the
  average. The other one filtered on `status = 'completed'`, which isn't a real status, so it returned nothing.
- "Names of users with an order over 300" - it gave `SELECT user_id FROM orders WHERE total > 300`. This one is the
  worst because it looks right. The filter is correct, but you get 21 ids with repeats instead of 15 names, and a
  user probably wouldn't notice.
- "Five most expensive orders" - there's no ORDER BY or LIMIT, so it made up filters. One was `total = 10000`,
  which got 0 rows, so a user might think there just aren't any orders. The other was a weird chain of
  `OR total >= ...` that returned all 50 orders in no particular order instead of the top 5.

So none of the constrained Group 2 answers were actually right, even when SQLite accepted them. The unconstrained
model could use JOIN, AVG and ORDER BY, so it got several of these right. The main thing the grammar fixed was
formatting. The unconstrained model sometimes wrapped its answer in markdown code fences, and that made SQLite
reject it.

**The grammar hard-codes the column names. What error does that rule out, and what breaks with any identifier?**

It rules out made-up or misspelled names. The model can't write `nmae`, `customer_name`, a table like
`customers`, or aliases like `o.total`. So "no such table" can't happen, and neither can "no such column" for a
name that doesn't exist at all.

It doesn't stop a real column from being used on the wrong table though, because `NAME` is one list for both
tables and the columns come before FROM. That's exactly what made the constrained Group 2 queries fail
(`name` from orders, `user_id` from users).

If `NAME` was any identifier like `/[a-z_]+/`, the model could make up any column name it wanted, and the grammar
would only be checking syntax. Words like `from` and `where` would also match as column names unless I excluded
them.

**Two things a context-free grammar can't check**

1. Whether a column actually belongs to the table in the FROM. The grammar only sees the text, not the schema, and
   the columns are written before the table. (I could hack it for two tables by writing a separate rule for each
   table, but that doesn't work for a schema that isn't known ahead of time.)
2. Whether the query means what the user asked. The grammar doesn't know what's in the data. For example, the
   model wrote `status = 'Cancelled'`, but the data says `cancelled` (lowercase), so it matched nothing. It also
   used statuses like `'completed'` and `'active'` that don't exist. All of these are valid SQL that SQLite runs
   fine, but they give the wrong answer.